In [2]:
import matplotlib.pyplot as plt

from phase_II.fast_wigner_function.smoothing_and_inverting_wigner_from_inference import invert_wigner_function
from phase_II.nifty_re_playground.useful.helpers import smooth_matrix
%matplotlib tk
import numpy as np
from useful.helpers import *
import nifty.nifty.re as jft
import jax.numpy as jnp
from scipy.optimize import curve_fit

In [ ]:
strain = np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_strain_values.txt") * 1e19
times = np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_time_values.txt")
times = times - times[0]

In [ ]:
plt.plot(times, strain)
usual_plot(xl="time in seconds")

In [ ]:
k, ps = power_analyze_re(times, strain)

first_negative_entry = np.where(k < 0)[0][0]

positive_k = k[1:first_negative_entry-1]  # all positive entries but the zero mode
positive_ps = ps[1:first_negative_entry-1]  # all positive entries but the zero mode

In [ ]:
plt.plot(positive_k, positive_ps)
plt.loglog()
usual_plot(xl="fourier modes", yl="power", title="Power spectrum sample")

Now, let's take a look at the correlated field model. We want to calculate the power spectrum as

$$ p_s(k) \propto e^{\gamma}, $$

where

$$ \gamma(l) = ml+\eta \int_{l_0}^l \int_{l_0}^{l^{\prime}} \xi_w(l^{\prime\prime
}) \: \mathrm{d}l^{\prime}\mathrm{d}l^{\prime\prime},$$

where $l_0$ is the natural logarithm of the first fourier mode greater than 0. Let's first understand the integral itself.

## Wiener process

$$ \mathrm{WP}(x) = \int_{x_0}^{x} \xi_w(x^{\prime
}) \:\mathrm{d}x^{\prime} $$

In [ ]:
def integrate_successively(x_array, xi_w):
    """
    :param x_array:         The time array over which xi_w was sampled.
    :param xi_w:            An array to integrate over.
    :return: An array consisting of cummulative integrals, increasing the upper bound x with each step
    """

    y0 = 0
    N = len(x_array)

    y = [y0]

    for i in range(1, N):
        x_prime = x_array[:i]
        xi_w_prime = xi_w[:i]

        res = np.trapz(xi_w_prime, x=x_prime)
        y.append(res)

    return jnp.array(y)

def wiener_process(x_array, custom_xi_w=None):
    """

    :param x_array:     The time array over which evolution occurs and over which a xi_w white noise variable is drawn.
    :param custom_xi_w: If none, a new white noise variable is drawn.
    :return:            The drawn white noise variable and the resulting wiener process
    """
    N = len(x_array)
    if custom_xi_w is None:
        xi_w = np.random.standard_normal(N)
    else:
        xi_w = custom_xi_w

    dx = x_array[1] - x_array[0]
    xi_w = xi_w/np.sqrt(dx)  # scaling due to discretization needed for wiener process to have variance proportional to t

    wp = integrate_successively(x_array, xi_w)
    return xi_w, wp


def integrated_wiener_process(x_array, custom_xi_w=None):
    """
    Integrates white noise twice cummulatively.
    :param x_array:  The time array over which evolution occurs and over which a xi_w white noise variable is drawn.
    :param custom_xi_w: If none, a new white noise variable is drawn.
    :return: The drawn white noise variable, the resulting wiener process and its integral.
    """
    xi_w, wp = wiener_process(x_array, custom_xi_w=custom_xi_w)
    iwp = integrate_successively(x_array, wp)
    return xi_w, wp, iwp

In [ ]:
x_array = np.linspace(0, 10, 300)

xi_w_tmp, wp_tmp = wiener_process(x_array)

plt.plot(x_array, xi_w_tmp, label="white noise")
plt.plot(x_array, wp_tmp, label="resulting wiener process")
usual_plot(yl="Arbitary units")

In [ ]:
xi_w_tmp, wp_tmp, iwp_tmp = integrated_wiener_process(x_array)

plt.plot(x_array, xi_w_tmp, label="white noise")
plt.plot(x_array, wp_tmp, label="Resulting wiener process")
plt.plot(x_array, iwp_tmp, label="Resulting integrated wiener process")
usual_plot(yl="Arbitary units")

In [ ]:
from typing import Optional

def correlated_field_ps(l, m, fluct, eta:Optional[jnp.array], xi_w:Optional[jnp.array]):
    """

    :param xi_w:    Whether to include stochastic iwp next to power law. None = don't include stochastic iwp.
                    If array-like, include stochastic iwp.
    :param l:       The log of positive fourier modes, except the zero-mode.
    :param m:       The power spectrum power law slope.
    :param fluct:   A to be set arbitrary norm corresponding to the fluctuations parameter.
    :return:        The correlated field power spectrum.
    """
    return fluct * jnp.exp(gamma(l=l, m=m, eta=eta, xi_w=xi_w))

def gamma(l, m, eta, xi_w):
    """

    :param l:               The log of positive fourier modes, except the zero-mode.
    :param m:               The power spectrum power law slope.
    :param eta:             The strength of the integrated wiener process component.
    :param xi_w:            If none, don't include stochastic iwp next to power law. If array-like, do.
    :return:                A model of the logarithm of the power spectrum. In this case, eta has to be provided.
    """
    power_law = m*l
    if xi_w is None:
        return power_law
    else:
        if eta is None:
            raise ValueError("Eta needs to be provided.")
        _, _, stochastic_component = integrated_wiener_process(x_array=l, custom_xi_w=xi_w)
        return power_law + eta * stochastic_component


def gamma_for_plotting(l, m, eta, iwp):
    power_law = m*l
    stochastic_component = iwp
    return power_law + eta * stochastic_component

def correlated_field_ps_for_plotting(l, m, eta, iwp, norm_power_law, fluct):
    gamma_realization = gamma_for_plotting(l=l, m=m, iwp=iwp, eta=eta)
    return fluct * jnp.exp(gamma_realization)


In [ ]:
log_modes = jnp.log(positive_k)

def find_cfm_parameters(function_to_minimize, p0, plot=True):
    """

    :param function_to_minimize:    The objective function to be minimized.
    :param p0:                      First guess on the parameters.
    :param plot:                    Whether to plot results.
    :return:
    """

    result = curve_fit(function_to_minimize, xdata=log_modes, ydata=positive_ps, p0=p0)
    mean_parameters = result[0]
    covariance = result[1]

    if plot:
        plt.plot(positive_k, positive_ps, label="Power spectrum")
        plt.plot(positive_k, function_to_minimize(log_modes, *mean_parameters), label="Curve fit approximation")
        plt.loglog()
        usual_plot(yl="Power", xl="Unique and positive k")

    print("Mean parameters found: ", *mean_parameters)
    print("\nCovariance matrix: \n", covariance)

    return mean_parameters, covariance


In [ ]:
obj_function_1 = lambda l, m, a: correlated_field_ps(l=l, m=m, fluct=a, xi_w=None, eta=None)
(curve_fit_m, curve_fit_fluct), _ = find_cfm_parameters(function_to_minimize=obj_function_1, plot=True, p0=(-2, 10))

In [ ]:
obj_function_2 = lambda l, eta, *xi_w: correlated_field_ps(l=l, m=curve_fit_m, fluct=curve_fit_fluct, eta=eta, xi_w=jnp.array(xi_w))

In [ ]:
iwp_parameters, _ = find_cfm_parameters(function_to_minimize=obj_function_2, plot=True, p0=np.random.standard_normal(len(log_modes)))

In [ ]:
# Unpack found parameters

In [ ]:
_, wp_from_fit, iwp_from_fit = integrated_wiener_process(log_modes, custom_xi_w=curve_fit_xi_w)

In [ ]:
plt.plot(log_modes, curve_fit_xi_w, label="white noise")
plt.plot(log_modes, wp_from_fit, label="Resulting wiener process")
plt.plot(log_modes, iwp_from_fit, label="Resulting integrated wiener process")
usual_plot(yl="Arbitary units")

In [ ]:
reduced_ps_for_iwp_plot = lambda iwp: correlated_field_ps_for_plotting(l=log_modes, m=curve_fit_slope,  iwp=iwp, global_norm=curve_fit_norm, norm_iwp=50, norm_power_law=1)

plt.plot(positive_k, positive_ps, label="Power spectrum")
plt.plot(positive_k, reduced_ps_for_iwp_plot(iwp_from_fit), label="Curve fit approximation")
plt.loglog()
usual_plot(yl="Power", xl="Unique and positive k")

## Before we continue...

If I smooth the Wigner function and run that through the inverse's power spectrum, do I get something that looks like the power spectrum sample of the waveform?

In [4]:
wigner_function_from_inference, t_inference, f_inference = unpickle_me_this("/Users/iason/PycharmProjects/STRAIN/phase_II/wigner_result_pipe_2.pickle")

In [ ]:
smoothed_wigner = jnp.abs(smooth_matrix(wigner_function_from_inference, smoothing_lvl=5, mode="gaussian"))

# from scipy.ndimage import binary_opening
#
# mask = smoothed_wigner > 5*np.std(smoothed_wigner.flatten())
# clean = binary_opening(mask, structure=np.ones((3,3)))

from scipy.ndimage import label

threshold = np.percentile(smoothed_wigner, 99.5)  # or manually tuned
mask = smoothed_wigner > threshold

# label connected regions
labeled, n = label(mask)
sizes = np.bincount(labeled.ravel())
keep = sizes > 50  # min pixel count
keep[0] = False
clean_mask = keep[labeled]

# optionally multiply by original mat to preserve amplitude
cleaned_wigner = smoothed_wigner * clean_mask


In [ ]:
print(5*np.std(smoothed_wigner.flatten()))

In [ ]:
visualize_stress(cleaned_wigner, cols=t_inference, rows=f_inference, smooth=False)

In [ ]:
wigner_gradient = np.gradient(jnp.abs(smoothed_wigner), axis=1)

In [ ]:
_ = plt.figure()
plt.plot(t_inference, np.sum(cleaned_wigner, axis=0))
usual_plot(yl="Frequency marginalized stress")

In [ ]:
print(np.mean(wigner_gradient, axis=1))

In [ ]:
np.cumsum

## Whitening a 2D Image with a Reference Power Spectrum

Why don't we just calculate the power spectrum of random white noise wigner once and then whiten with that?


**Goal:** We want to normalize the frequency content of an image `A` using the power spectrum of a reference image `B`.
Since FFT assumes periodicity, we first apply a 2D Tukey window to reduce edge artifacts.
After windowing, we can safely compute Fourier transforms and perform spectral whitening.


In [5]:
from scipy.signal.windows import tukey

white_noise = np.random.standard_normal(len(t_inference))
white_noise_tukeyed = tukey(len(white_noise), alpha=0.1, sym=True) * white_noise

In [6]:
white_noise_stress, _, _ = Stress_re(white_noise_tukeyed, time=t_inference)


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done






/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/useful/helpers.py:1163: UserWarning: Realness threshold was not passed. Mean imaginary part of stress field larger than 1e-10 (1.2588453890405304e-10).
  raise_warning(


In [8]:
hist_data = plt.hist(white_noise_stress.flatten().real, bins=100)
plt.show()

In [15]:
data = white_noise_stress.flatten().real[::100]

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, t


# Histogram
counts, bin_edges = np.histogram(data, bins=50)
bin_centers = 0.5*(bin_edges[1:] + bin_edges[:-1])

# Gaussian fit to data
mu, std = norm.fit(data)
pdf_gauss = norm.pdf(bin_centers, mu, std)

# Student-t fit to data (heavy-tailed)
df, loc, scale = t.fit(data)
pdf_t = t.pdf(bin_centers, df, loc, scale)

# Normalize PDFs to histogram
pdf_gauss_scaled = pdf_gauss * np.sum(counts) * (bin_edges[1]-bin_edges[0])
pdf_t_scaled = pdf_t * np.sum(counts) * (bin_edges[1]-bin_edges[0])

# Plot
plt.figure(figsize=(10,5))
plt.bar(bin_centers, counts, width=bin_edges[1]-bin_edges[0], alpha=0.5, label='Histogram')
plt.plot(bin_centers, pdf_gauss_scaled, 'r-', lw=2, label='Gaussian fit')
plt.plot(bin_centers, pdf_t_scaled, 'g--', lw=2, label='Student-t fit')
plt.yscale('log')
plt.xlabel('Value')
plt.ylabel('Counts')
plt.title('Histogram with Gaussian and Student-t fits')
plt.legend()
plt.show()


In [19]:
import numpy as np
from itertools import product
from scipy.ndimage import gaussian_filter

def taylor_expand_image(img, x0, y0, order=2, dx=1.0):
    """
    Compute Taylor expansion of a 2D image around (x0, y0) up to given order.

    Parameters
    ----------
    img : 2D ndarray
        Input image.
    x0, y0 : int
        Pixel coordinates around which to expand.
    order : int
        Maximum order of the Taylor expansion.
    dx : float
        Grid spacing (pixel size); assume square pixels.

    Returns
    -------
    img_taylor : 2D ndarray
        Taylor-expanded image approximation.
    """
    ny, nx = img.shape
    img_taylor = np.zeros_like(img, dtype=float)

    # Prepare coordinate grids relative to (x0, y0)
    X, Y = np.meshgrid(np.arange(nx) - x0, np.arange(ny) - y0)

    # Precompute derivatives using central finite differences
    derivatives = {}

    # Zeroth order
    derivatives[(0,0)] = img[y0, x0]

    for total_order in range(1, order+1):
        for i, j in product(range(total_order+1), repeat=2):
            if i + j != total_order:
                continue
            # Finite difference approximation for derivative of order (i,j)
            deriv = img.copy()
            for _ in range(i):
                deriv = np.gradient(deriv, dx, axis=1)
            for _ in range(j):
                deriv = np.gradient(deriv, dx, axis=0)
            derivatives[(i,j)] = deriv[y0, x0]

    # Construct Taylor expansion
    for (i,j), val in derivatives.items():
        img_taylor += val * (X**i) * (Y**j) / (np.math.factorial(i) * np.math.factorial(j))

    return img_taylor

import matplotlib.pyplot as plt

# Example image: smooth 2D Gaussian
nx, ny = 50, 50
x = np.linspace(-1,1,nx)
y = np.linspace(-1,1,ny)
X, Y = np.meshgrid(x, y)
img = np.exp(-(X**2 + Y**2))

# 2nd-order Taylor around center
img_taylor = taylor_expand_image(img, x0=nx//2, y0=ny//2, order=4)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(img, origin='lower')
plt.title("Original Image")
plt.colorbar()

plt.subplot(1,2,2)
plt.imshow(img_taylor, origin='lower')
plt.title("2nd-order Taylor Expansion")
plt.colorbar()
plt.show()



/var/folders/6s/zt639pwd3kg46kcb8b3t121r0000gp/T/ipykernel_53873/2201610725.py:51: DeprecationWarning: `np.math` is a deprecated alias for the standard library `math` module (Deprecated Numpy 1.25). Replace usages of `np.math` with `math`
  img_taylor += val * (X**i) * (Y**j) / (np.math.factorial(i) * np.math.factorial(j))


In [26]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

def taylor_expand_order(img, x0, y0, order=0, dx=1.0):
    """
    Compute the Taylor expansion contribution of a specific order only.
    """
    ny, nx = img.shape
    X, Y = np.meshgrid(np.arange(nx)-x0, np.arange(ny)-y0)
    contribution = np.zeros_like(img, dtype=float)

    if order == 0:
        return np.full_like(img, img[y0, x0])

    # Compute derivatives for this order
    for i,j in product(range(order+1), repeat=2):
        if i+j != order:
            continue
        deriv = img.copy()
        for _ in range(i):
            deriv = np.gradient(deriv, dx, axis=1)
        for _ in range(j):
            deriv = np.gradient(deriv, dx, axis=0)
        contribution += deriv[y0, x0] * (X**i)*(Y**j) / (np.math.factorial(i)*np.math.factorial(j))

    return contribution

def plot_taylor_sum(img, N, dx=1.0):
    """
    Plot original image (left) and sum of first N Taylor orders (right)
    """
    ny, nx = img.shape
    # x0, y0 = nx//2, ny//2  # center
    x0, y0 = 5752, 153

    # Sum contributions
    taylor_sum = np.zeros_like(img, dtype=float)
    for order in range(N+1):
        taylor_sum += taylor_expand_order(img, x0, y0, order=order, dx=dx)

    # Plot
    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1)
    plt.imshow(img, origin='lower', cmap='viridis')
    plt.title("Original Image")
    plt.colorbar()

    plt.subplot(1,2,2)
    plt.imshow(taylor_sum, origin='lower', cmap='viridis')
    plt.title(f"Sum of first {N} Taylor orders")
    plt.colorbar()

    plt.tight_layout()
    plt.show()

# Example usage:
plot_taylor_sum(smooth_matrix(wigner_function_from_inference,5).real, N=10)


/var/folders/6s/zt639pwd3kg46kcb8b3t121r0000gp/T/ipykernel_53873/130164415.py:25: DeprecationWarning: `np.math` is a deprecated alias for the standard library `math` module (Deprecated Numpy 1.25). Replace usages of `np.math` with `math`
  contribution += deriv[y0, x0] * (X**i)*(Y**j) / (np.math.factorial(i)*np.math.factorial(j))


In [11]:
visualize_stress(white_noise_stress, cols=t_inference, rows=f_inference, smooth=True)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


In [27]:
from scipy.signal.windows import tukey

white_noise = np.random.standard_normal(len(f_inference))
white_noise_prime = 5*np.random.standard_normal(len(f_inference))

white_noise_stress, _, _ = Stress_re(white_noise, time=t_inference)
white_noise_stress_prime, _, _ = Stress_re(white_noise_prime, time=t_inference)

A = white_noise_stress - np.mean(white_noise_stress.flatten())

# B = smooth_matrix(white_noise_stress_prime, smoothing_lvl=5, mode="gaussian")
wigner_function_from_inference_ordered = np.fft.fftshift(wigner_function_from_inference, axes=0)
freqs_ordered = np.fft.fftshift(f_inference, axes=0)
C = wigner_function_from_inference_ordered


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done





Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix


/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/useful/helpers.py:1163: UserWarning: Realness threshold was not passed. Mean imaginary part of stress field larger than 1e-10 (1.109001918075947e-10).
  raise_warning(


	 ... Done






/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/useful/helpers.py:1163: UserWarning: Realness threshold was not passed. Mean imaginary part of stress field larger than 1e-10 (3.2198432808883126e-09).
  raise_warning(


In [10]:
def add_gaussian_blob(image, center, sigma=3, amplitude=1.0):
    """
    Add a small 2D Gaussian blob to a 2D image.

    Parameters
    ----------
    image : 2D np.ndarray
        The image to modify
    center : tuple of int
        (y0, x0) center of the Gaussian
    sigma : float
        Standard deviation of the Gaussian in pixels
    amplitude : float
        Peak value of the Gaussian
    """
    ny, nx = image.shape
    y = np.arange(ny)[:, None]
    x = np.arange(nx)[None, :]
    y0, x0 = center

    blob = amplitude * np.exp(-((x - x0)**2 + (y - y0)**2) / (2 * sigma**2))
    image_with_blob = image + blob
    return image_with_blob

A_gaussian_blob = add_gaussian_blob(A, center=(4000, 2000), sigma=30, amplitude=1)

# # Parameters for the stripe
# x_pos = 100          # column index where the stripe appears
# width = 100            # number of columns
# amplitude = 1e3      # value to add
#
# # Add vertical stripe
# A_with_vertical_line[300:2000, x_pos:x_pos+width] += amplitude*np.cos(np.linspace(0, 2*np.pi, 100))
# A_with_vertical_line = A_with_vertical_line.T


In [28]:
# visualize_stress(A, cols=t_inference, rows=freqs_ordered, smooth=True)
# visualize_stress(A_gaussian_blob, cols=t_inference, rows=freqs_ordered, smooth=False)
# visualize_stress(B, cols=t_inference, rows=f_inference, smooth=False)
visualize_stress(C, cols=t_inference, rows=freqs_ordered, smooth=False)

### Let's window

In [29]:
def tukey_window_this(mat, alpha=0.2):

    ny, nx = mat.shape
    wx = tukey(nx, alpha=alpha)
    wy = tukey(ny, alpha=alpha)

    # Make 2D Tukey window
    window2d = wy[:, None] * wx[None, :]

    return mat * window2d

# Apply window to images
A_win = tukey_window_this(A)
# A_with_line_win = tukey_window_this(A_with_vertical_line)
# A_blob_window = tukey_window_this(A_gaussian_blob)
# B_win = tukey_window_this(B)
C_win = tukey_window_this(C)

In [ ]:
# Optional: visualize
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(A_win.real, origin='lower', cmap='viridis')
plt.title("Windowed A")
plt.colorbar()
plt.subplot(1,2,2)
plt.imshow(A_blob_window.real, origin='lower', cmap='viridis')
plt.title("Windowed B or C")
plt.colorbar()
plt.show()

In [31]:
def power_spectrum_2d_sample(image):
    F = np.fft.fft2(image)         # 2D FFT
    # F = np.fft.fftshift(F)         # shift zero freq to center
    P = np.abs(F)**2               # squared magnitude = power
    return P

def whiten_image(image: np.ndarray, psd2d: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    # FFT of the input image
    F = np.fft.fft2(image)

    # Whitening: divide Fourier amplitudes by sqrt(PSD)
    F_whitened = F / (np.sqrt(psd2d) + eps)

    # Back to real space
    image_whitened = np.fft.ifft2(F_whitened).real
    return image_whitened

In [32]:
to_avg = []
for _ in range(10):
    white_noise = 0.5 * np.random.standard_normal(len(f_inference))
    white_noise_stress, _, _ = Stress_re(white_noise, time=t_inference, supress_print=True)
    # A = smooth_matrix(white_noise_stress, smoothing_lvl=5, mode="gaussian")
    A = white_noise_stress
    A_win = tukey_window_this(A)
    ps_sample = power_spectrum_2d_sample(A_win)
    to_avg.append(ps_sample)


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done

Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done

Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done

Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done

Calculating stress...
	

In [33]:
white_noise_wigner_psd = np.mean(np.array(to_avg), axis=0)

In [ ]:
plt.figure(figsize=(10, 6))
plt.imshow(white_noise_wigner_psd, aspect='auto', origin='lower', cmap='jet')
plt.colorbar(label='PSD magnitude')
plt.xlabel('Frequency index')
plt.ylabel('Time index')
plt.title('Wigner PSD of White Noise')
plt.show()

In [ ]:
print(np.min(white_noise_wigner_psd))

In [34]:
# A_whitened = whiten_image(A_win, white_noise_wigner_psd)
# A_with_line_whitened = whiten_image(A_with_line_win, white_noise_wigner_psd, eps=0)
# A_with_blob_whitened = whiten_image(A_blob_window, white_noise_wigner_psd, eps=0)
# B_whitened = whiten_image(B_win, white_noise_wigner_psd)
C_whitened = whiten_image(C_win, white_noise_wigner_psd, eps=0)

In [36]:
# visualize_stress(A_whitened, cols=t_inference, rows=freqs_ordered, smooth=True)
# visualize_stress(B_whitened, cols=t_inference, rows=f_inference, smooth=False)
visualize_stress(C_whitened, cols=t_inference, rows=freqs_ordered, smooth=True)

In [37]:
visualize_stress(C, cols=t_inference, rows=freqs_ordered, smooth=True)

In [54]:
import numpy as np
import matplotlib.pyplot as plt

def subtract_and_floor(A, B, N=1):
    """
    Repeatedly subtract A from B N times, flooring at 0.
    """
    result = B.copy()
    for _ in range(N):
        result = np.maximum(result - A, 0)
    return result

def plot_subtraction(A, B, N=1):
    """
    Plot original B on the left and B after N subtractions of A on the right.
    """
    B_sub = subtract_and_floor(A, B, N)

    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1)
    plt.imshow(B, origin='lower', cmap='viridis')
    plt.title("Original B")
    plt.colorbar()

    plt.subplot(1,2,2)
    plt.imshow(B_sub, origin='lower', cmap='viridis')
    plt.title(f"B after {N} subtractions of A")
    plt.colorbar()

    plt.tight_layout()
    plt.show()

# Example usage:
C_whitened_smoothed = smooth_matrix(C_whitened, 5).real
C_smoothed = smooth_matrix(C, 5).real
C_smoothed /= np.max(C_smoothed)

plot_subtraction(C_smoothed, C_whitened_smoothed, N=2)


In [46]:
print(np.max(C_whitened_smoothed))
np.max(C_smoothed)

2.3833794289182477


4191.1880088945545

In [ ]:
_ = plt.figure()
plt.plot(t_inference, np.sum(A_with_blob_whitened, axis=0))
usual_plot(yl="Frequency marginalized stress")

$$ d_{wh} = iFFT ((\tilde{n}+\tilde{s})/\sqrt{p_n})$$

so

$$<\tilde{d}_w \tilde{d}_w^{\ast}> = 1 + <\tilde{s}^2>/p_n$$

so if $p_s$ larger than $p_n$ it stands out and if $p_s$ is 0 the background is white noise.

In [ ]:
import numpy as np

def stationarity_score(image, patch_size=50):
    """
    Estimate how stationary a 2D image is.

    Parameters
    ----------
    image : 2D np.ndarray
        Input image
    patch_size : int
        Size of square patches to analyze

    Returns
    -------
    score : float
        Relative variation of local means and variances; smaller is more stationary
    details : dict
        Contains std/mean of local means and variances
    """
    ny, nx = image.shape
    local_means = []
    local_vars = []

    for i in range(0, ny, patch_size):
        for j in range(0, nx, patch_size):
            patch = image[i:i+patch_size, j:j+patch_size]
            local_means.append(patch.mean())
            local_vars.append(patch.var())

    local_means = np.array(local_means)
    local_vars = np.array(local_vars)

    mean_rel_std = np.std(local_means) / (np.std(image) + 1e-12)
    var_rel_std  = np.std(local_vars)  / (np.var(image) + 1e-12)

    score = mean_rel_std + var_rel_std
    details = {
        "mean_rel_std": mean_rel_std,
        "var_rel_std": var_rel_std
    }

    return score, details

# -----------------------------
# Unit test with white noise
# -----------------------------
ny, nx = 512, 512
white_noise = np.random.standard_normal((ny, nx))

score, details = stationarity_score(white_noise, patch_size=64)

print("Stationarity score:", score)
print("Details:", details)


In [ ]:
score, details = stationarity_score(smooth_matrix(white_noise_stress, 5), patch_size=64)

print("Stationarity score:", score)
print("Details:", details)

In [ ]:
plt.figure(figsize=(10, 6))
plt.imshow(power_spectrum_2d_sample(A_whitened), aspect='auto', origin='lower', cmap='jet')
plt.colorbar(label='PSD magnitude')
plt.xlabel('Frequency index')
plt.ylabel('Time index')
plt.title('Wigner PSD of White Noise')
plt.show()

## 2nd test

In [1]:
import numpy as np

def radial_psd(image, nbins=50):
    ny, nx = image.shape
    fy = np.fft.fftfreq(ny)
    fx = np.fft.fftfreq(nx)
    kx, ky = np.meshgrid(fx, fy)
    k = np.sqrt(kx**2 + ky**2)

    F = np.fft.fft2(image)
    P2d = np.abs(F)**2

    # Bin radially
    k_flat = k.ravel()
    P_flat = P2d.ravel()
    k_bins = np.linspace(0, k_flat.max(), nbins+1)
    P_radial = np.zeros(nbins)
    for i in range(nbins):
        mask = (k_flat >= k_bins[i]) & (k_flat < k_bins[i+1])
        if np.any(mask):
            P_radial[i] = P_flat[mask].mean()
    k_centers = 0.5*(k_bins[:-1] + k_bins[1:])
    return k_centers, P_radial


In [7]:
def map_2d_to_radial_psd(nx, ny, k_centers, P_radial):
    fy = np.fft.fftfreq(ny)
    fx = np.fft.fftfreq(nx)
    kx, ky = np.meshgrid(fx, fy)
    k2d = np.sqrt(kx**2 + ky**2)

    # Interpolate radial PSD onto 2D k-grid
    from scipy.interpolate import interp1d
    interp = interp1d(k_centers, P_radial, bounds_error=False, fill_value=P_radial[-1])
    P2d_iso = interp(k2d)
    return P2d_iso

def whiten_image_radial(image, P2d_iso, eps=1e-12):
    F = np.fft.fft2(image)
    F_whitened = F / (np.sqrt(P2d_iso) + eps)
    return np.fft.ifft2(F_whitened).real


In [12]:
# Estimate radial PSD from noise images

im_to_analyze = A_win

k_centers, P_radial = radial_psd(im_to_analyze)
P2d_iso = map_2d_to_radial_psd(*im_to_analyze.shape, k_centers, P_radial)

im_to_analyze_radially_whitened = whiten_image_radial(im_to_analyze, P2d_iso)

In [14]:
plt.loglog(k_centers, P_radial)
plt.show()

In [16]:
visualize_stress(im_to_analyze_radially_whitened, rows=freqs_ordered, cols=t_inference, smooth=True)

In [57]:
invert_wigner_function(2)

NameError: name 'invert_wigner_function' is not defined